# OpenAI Responses API — Practical Notebook

This practical notebook covers the key concepts from the lesson on the **OpenAI Responses API**:

1. Setting up the OpenAI Python SDK
2. Making a simple Responses API call
3. Understanding simple input and output
4. Conversation chaining with `previous_response_id`
5. Using built-in tools such as Web Search
6. Understanding model-driven tool usage
7. Comparing Responses API with the older Chat Completions style
8. Practical exercises and recap

> **Note:** Some examples require a valid OpenAI API key and may incur API usage costs.

## 1. What is the Responses API?

The **Responses API** is OpenAI's modern API interface for building applications that interact with models and can use tools.

It provides a simpler interface for:

- Sending user input to a model
- Receiving generated output
- Maintaining conversation context
- Calling built-in tools
- Calling custom functions
- Building agent-style applications

A simplified mental model is:

```text
Application
    |
    v
Responses API
    |
    v
OpenAI Model
    |
    +--> Optional Tools
          - Web Search
          - File Search
          - Code Interpreter
          - Custom Functions
```

The main idea is that the API provides a common interface for model responses and tool-enabled workflows.

## 2. Install the OpenAI Python Package

Run the following command in a terminal or notebook environment:

In [ ]:
%pip install -U openai

## 3. Configure Your API Key

The OpenAI Python SDK can read your API key from the `OPENAI_API_KEY` environment variable.

### Windows PowerShell

```powershell
$env:OPENAI_API_KEY="your-api-key"
```

### macOS / Linux

```bash
export OPENAI_API_KEY="your-api-key"
```

For production applications, avoid hard-coding API keys directly in source code.

In [ ]:
import os
from openai import OpenAI

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is not set. Configure it before running API examples.")

client = OpenAI()

print("OpenAI client initialized successfully.")

## 4. Simplest Responses API Call

The basic pattern is straightforward:

1. Create an OpenAI client.
2. Call `client.responses.create()`.
3. Specify a model.
4. Provide the input.
5. Read the generated text using `response.output_text`.

In [ ]:
response = client.responses.create(
    model="gpt-4.1",
    input="Explain what an AI agent is in two simple sentences."
)

print(response.output_text)

### Explanation

The important part is the simplicity of the interface:

```python
response = client.responses.create(
    model="gpt-4.1",
    input="Your question here"
)
```

The generated text can then be accessed directly:

```python
response.output_text
```

This avoids manually navigating deeply nested response structures for the common case of retrieving generated text.

## 5. A More Structured Input

Although a simple string is convenient, the Responses API can also work with structured input. This becomes useful when building applications with richer instructions or multiple input items.

For example, you can separate instructions from the user's request.

In [ ]:
response = client.responses.create(
    model="gpt-4.1",
    instructions="You are a concise Python programming tutor.",
    input="Explain the difference between a list and a tuple in Python."
)

print(response.output_text)

## 6. Conversation Chaining with `previous_response_id`

One of the useful features of the Responses API is the ability to connect responses using `previous_response_id`.

Conceptually:

```text
User message 1
      |
      v
Response 1
      |
      | previous_response_id
      v
User message 2
      |
      v
Response 2
```

This allows a follow-up request to reference the previous response in the conversation chain.

In [ ]:
# First turn
response_1 = client.responses.create(
    model="gpt-4.1",
    input="My name is Alice. I am a software engineer and I love hiking."
)

print("First response:")
print(response_1.output_text)

# Second turn, linked to the first response
response_2 = client.responses.create(
    model="gpt-4.1",
    input="What is my name and what is my hobby?",
    previous_response_id=response_1.id
)

print("\nSecond response:")
print(response_2.output_text)

### What happened?

The second request references the first response using:

```python
previous_response_id=response_1.id
```

The model can use the linked conversation context to answer the follow-up question.

This is especially useful for multi-turn applications where each new turn builds on previous interactions.

## 7. Building a Small Multi-Turn Conversation

The same technique can be repeated for multiple turns.

In [ ]:
turn_1 = client.responses.create(
    model="gpt-4.1",
    input="I am learning Python and I want to become a backend developer."
)

turn_2 = client.responses.create(
    model="gpt-4.1",
    input="What should I learn first?",
    previous_response_id=turn_1.id
)

turn_3 = client.responses.create(
    model="gpt-4.1",
    input="Can you give me a 30-day learning plan?",
    previous_response_id=turn_2.id
)

print("Turn 1:", turn_1.output_text)
print("\nTurn 2:", turn_2.output_text)
print("\nTurn 3:", turn_3.output_text)

## 8. Built-in Tools

The Responses API can also work with built-in tools.

Common examples include:

- Web Search — retrieve current information from the web
- File Search — search information from uploaded files/vector stores
- Code Interpreter — execute code for tasks such as analysis
- Computer Use — interact with computer interfaces in supported scenarios
- Image Generation — generate images through supported model capabilities

The important concept is **model-driven tool usage**. When a tool is available, the model can determine when it is useful instead of the application having to manually decide every time.

## 9. Web Search Example

The following example demonstrates the built-in web search tool.

> Tool availability and exact model/tool support can change over time. If this example does not work with your account or selected model, check the current OpenAI API documentation and supported models.

In [ ]:
response = client.responses.create(
    model="gpt-4.1",
    tools=[{"type": "web_search_preview"}],
    input="What are the latest major developments in AI this week?"
)

print(response.output_text)

### Understanding Model-Driven Search

Consider two questions:

1. **What is the capital of France?**
2. **What are the latest AI developments this week?**

The first question is stable general knowledge, so a web search may not be necessary.

The second question is time-sensitive. A web search is useful because the answer depends on current information.

The key idea is that the model can determine whether the enabled tool is useful for the request.

## 10. Simple Question Without Web Search

The same API can answer a general knowledge question without requiring an external tool.

In [ ]:
response = client.responses.create(
    model="gpt-4.1",
    tools=[{"type": "web_search_preview"}],
    input="What is the capital of France?"
)

print(response.output_text)

## 11. Comparing the Mental Model with Older Chat Completions

A simplified older Chat Completions style often looked like this:

```python
response = client.chat.completions.create(
    model="gpt-4.1",
    messages=[
        {"role": "user", "content": "Hello"}
    ]
)

answer = response.choices[0].message.content
```

The Responses API provides a more unified interface for model output and tool-enabled workflows:

```python
response = client.responses.create(
    model="gpt-4.1",
    input="Hello"
)

answer = response.output_text
```

The exact APIs and capabilities evolve, but the practical lesson is that the Responses API is designed as a central interface for modern OpenAI model interactions.

## 12. Practical Example — Research Assistant

Let's create a small helper function that uses the Responses API.

In [ ]:
def ask_ai(question, use_web=False):
    kwargs = {
        "model": "gpt-4.1",
        "input": question,
    }

    if use_web:
        kwargs["tools"] = [{"type": "web_search_preview"}]

    response = client.responses.create(**kwargs)
    return response.output_text


print(ask_ai("Explain what an API is to a beginner."))
print("\n--- Current information ---\n")
print(ask_ai("What are the latest developments in AI?", use_web=True))

## 13. Practical Exercise 1 — Basic Response

**Task:** Modify the following code so the model explains Python decorators in three simple bullet points.

In [ ]:
# TODO: Complete this exercise

response = client.responses.create(
    model="gpt-4.1",
    input="Explain Python decorators in three simple bullet points."
)

print(response.output_text)

## 14. Practical Exercise 2 — Conversation Chaining

**Task:**

1. Tell the model your preferred programming language.
2. Ask it to recommend a project using that language.
3. Ask for a step-by-step implementation plan using `previous_response_id`.

Try to keep the conversation linked across all three responses.

In [ ]:
# TODO: Complete the three-turn conversation

first = client.responses.create(
    model="gpt-4.1",
    input="My preferred programming language is Python."
)

second = client.responses.create(
    model="gpt-4.1",
    input="Recommend one practical project for me.",
    previous_response_id=first.id
)

third = client.responses.create(
    model="gpt-4.1",
    input="Now give me a step-by-step implementation plan for that project.",
    previous_response_id=second.id
)

print("First:", first.output_text)
print("\nSecond:", second.output_text)
print("\nThird:", third.output_text)

## 15. Practical Exercise 3 — Model-Driven Web Search

**Task:** Ask a current, time-sensitive question and enable web search.

Examples:

- What are the latest Python releases?
- What are the latest developments in AI agents?
- What are the latest major technology announcements?

The goal is to observe how enabling a tool allows the model to use external information when appropriate.

In [ ]:
# TODO: Replace the question with your own current-information question

response = client.responses.create(
    model="gpt-4.1",
    tools=[{"type": "web_search_preview"}],
    input="What are the latest developments in AI agents?"
)

print(response.output_text)

## 16. Key Takeaways

### Responses API

The Responses API is a modern interface for interacting with OpenAI models and building tool-enabled applications.

### Simpler Input and Output

For basic requests, you can provide a string as input and access generated text through `output_text`.

### Conversation Chaining

Using `previous_response_id`, you can connect responses into a conversation chain and build multi-turn applications.

### Built-in Tools

Tools such as web search can be enabled through the API, allowing the model to use external capabilities when appropriate.

### Model-Driven Tool Usage

The model can determine when an enabled tool is useful instead of the application manually handling every tool decision.

### Important Architecture

```text
Your Application
       |
       v
Responses API
       |
       +------> OpenAI Model
       |
       +------> Built-in Tools
       |
       +------> Custom Tools / Functions
```

The major lesson is that the Responses API provides a unified foundation for building modern OpenAI-powered applications and agentic workflows.